# LabOne Q — Extra Lesson: Fast Rabi Tune-up on a Heterogeneous Control System

> **Emulation only, ZQCS driver required.** This lesson uses the HQCS programming model of LabOne Q and the
> emulated coprocessor from `zhinst_zqcs.emulation`, which ships with a ZQCS installation and is not on PyPI.
> Like `07`, it does not run on the open release pinned in `requirements.txt`; the outputs are kept in the
> notebook so the result can be read without running it.

This tutorial illustrates the LabOne Q programming model for a heterogeneous control system by walking you through a quantum-classical experiment: a Rabi tune-up in which a Bayesian optimizer, running on a classical coprocessor, refines the drive pulse while the experiment is running. You will declare the coprocessor and the data streams, write the pulse sequence, build the kernel that runs on the coprocessor, and read out the optimization trace from the results. Before you start, read Quantum_classical Algorithms and Heterogeneous Control for the high-level overview and The HQCS Programming Model for the concepts and DSL constructs used here.

Both the ZQCS and the coprocessor are **emulated**. The example serves as an educational example to illustrate the programming model in LabOne Q. Programming a _real_ Bayesian optimizer that runs on a physical processor attached to a real ZQCS will require a different definition of the coprocessor program than the one shown here in Python. The LabOne Q side of the example is realistic, however. 

**Limitations of the emulated setup**

We emulate both ZQCS and the coprocessor. The emulated models of both of these are intentionally limited. Notably:

- LabOne Q's emulated ZQCS produces synthetic measurement data (`(42 + qudit_index) + result_index * 1j`).
  Naturally, these data are not suitable for feeding into a real-world optimizer.
- The emulated coprocessor never actually applies a waveform update. Emulation mode does not support this.

The companion tutorial The Emulated Coprocessor covers the emulated coprocessor itself: how a kernel receives and emits packets, how the shape of the logged results follows the loop structure, and which violations of the contract the runtime reports.

We start by defining a [DeviceSetup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html) and a [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html). We also instantiate the emulated coprocessor.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from laboneq.dsl.experiment.builtins import *  # noqa: F403
from laboneq.simple import *  # noqa: F403
from zhinst_zqcs.emulation import EmulatedCoprocessorKernel, add_emulated_coprocessor

descriptor = """\
instruments:
  ZQCS:
  - address: http://DEV_ZQCS
    uid: zqcs
connections:
  zqcs:
    - iq_signal: q0/drive_line
      port: "1:3:1:1"
    - acquire_signal: q0/acquire_line
      port: "1:3:1:2"
    - iq_signal: q0/measure_line
      port: "1:3:1:3"
"""

setup = DeviceSetup.from_descriptor(
    yaml_text=descriptor, server_host="localhost", server_port=-1
)
emulated_coprocessor = add_emulated_coprocessor(setup)

session = Session(setup)
session.connect(do_emulation=True)

## The Experiment

A conventional Rabi experiment sweeps the amplitude of a drive pulse over a fixed grid, fits the resulting oscillation offline, and repeats until the fit converges. A Bayesian tune-up replaces the fixed grid with a closed loop: every shot's measurement result streams to a coprocessor, which maintains a probability distribution over the pulse parameters. After each batch of shots, the optimizer proposes a new pulse shape based on everything it has measured so far, and writes it back into the running experiment. The loop reaches a given precision with far fewer shots than a grid sweep, because each batch is placed where it is most informative. For the technique itself and the measurement-efficiency gains it delivers, see the demonstrations of Bayesian closed-loop calibration in [[1](#ref1), [2](#ref2), [3](#ref3), [4](#ref4)].

The experiment iterates over `N_CYCLES` optimization steps. In each step:

1. The ZQCS plays the current Rabi pulse and measures the qubit, `N_SHOTS` times.
2. Streams each shot's integrated result to the coprocessor.
3. After it receives all `N_SHOTS`, the coprocessor averages the results from the individual shots and updates its own estimate of the qubit state. During this time, the ZQCS is instructed to wait.
4. The coprocessor sends an updated waveform or amplitude to ZQCS for the next batch of `N_SHOTS`.
5. The steps above repeat for `N_CYCLES` rounds.


One optimization step, in words: the ZQCS plays and measures the Rabi pulse `N_SHOTS` times, and `send` streams each shot's result to the coprocessor as one packet while the sequence keeps running. The optimizer accumulates the batch and computes a waveform proposal. In the `wait_for_update` section, `mark_stale` opens the acceptance window, the 10 µs `delay` budgets the round trip, and `ensure_live` confirms that the proposal arrived; the next batch then plays the updated pulse, whose samples were replaced in place.

The supported types, directions, and binding methods of schema fields are listed in the HQCS Struct Reference, and HQCS Variables Reference gives an overview of LabOne Q `Variables`.

In [ ]:
N_CYCLES = 20
N_SHOTS = 100

rabi_pulse = pulse_library.gaussian(uid="rabi_drive", length=50e-9, amplitude=0.5)
measure_pulse = pulse_library.const(uid="measure", length=400e-9, amplitude=0.3)
readout_weights = pulse_library.const(length=400e-9)


@experiment(signals=["drive", "measure", "acquire"], coprocessors=["optimizer"])
def bayesian_rabi():

    # DECLARING THE STREAMS
    # The experiment needs one stream in each direction. The outbound stream carries the
    # integrated result of each shot; binding the acquisition handle `"rabi_shots"` means
    # every matching `acquire` feeds the stream without further commands in the sequence.
    # The inbound stream carries the optimizer's response: the proposed waveform, and the
    # current estimate of the Rabi amplitude, which you log to follow the convergence.
    # The inbound fields must be bound to the DSL objects they update. The waveform field
    # targets the pulse itself; the rabi_amplitude field becomes a logged `Variable`:
    shots = register_stream(
        dst="optimizer", schema=Struct(meas=IqDataPacked), uid="shots"
    )
    shots.fields["meas"].set_handles(["rabi_shot"])

    updates = register_stream(
        src="optimizer",
        schema=Struct(updated_waveform=WaveformUpdate, rabi_amplitude=Amplitude),
        uid="updates",
    )

    updates.fields["updated_waveform"].updates(rabi_pulse)
    amplitude = updates.fields["rabi_amplitude"].as_variable(name="rabi_amplitude")
    amplitude.log(handle="chosen_amplitude")

    # CREATING THE PULSE SEQUENCE
    # The sequence is an ordinary LabOne Q experiment with three additions: `send` after each
    # acquisition, `mark_stale` before each batch, and the `delay` plus `ensure_live` pair that
    # gives the optimizer its time budget.
    with acquire_loop_rt(
        count=N_CYCLES,
        uid="optimizer_cycles",
        averaging_mode=AveragingMode.SINGLE_SHOT,
        acquisition_type=AcquisitionType.INTEGRATION,
    ):
        with for_each(range(N_SHOTS), uid="shots", axis_name="shot"):
            with section(uid="drive_qubit"):
                play("drive", rabi_pulse)
            with section(uid="readout", play_after="drive_qubit"):
                play("measure", measure_pulse)
                acquire("acquire", handle="rabi_shot", kernel=readout_weights)
                delay(signal="acquire", time=2e-9)

            send(shots)

        mark_stale(rabi_pulse)  # acceptance window open: from here on, the waveform update may arrive
        mark_stale(amplitude)  # acceptance window open: from here on, the amplitude update may arrive
        with do_until(is_live(rabi_pulse), max_count=5):
            delay("drive", 10e-6)  # time budget for the optimizer to compute and respond.

Note that:

- `send(measurement_stream)` sits directly after the acquisition. It marks one shot's result as one packet; the transmission itself happens in the background while the sequence continues. The schema has no send-time fields in this example, so `send` takes no further arguments.
- The `delay` of 10 $\mu s$ is the explicit time budget for the full round trip: transport to the coprocessor, the Bayesian update, and the return of the waveform. `ensure_live(rabi_pulse)` then asserts that the proposal has arrived. If it has not, the experiment stops with an error instead of silently playing a stale pulse – see What We Guarantee. What You Own..

**Intuition: Why am I delaying my drive signal line?**

Remember that the programming model is pulse-centric: The timing of the pulse sheet is still entirely governed by our [timing rules](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/01_timing_rules.html), and classical-processing tasks run in the background. So the timing must be anchored via "real" signal lines. In the example here, we chose the drive line, but you could equally well have written the experiment by delaying the `measure` or `acquire` lines. Think about it like this: you need to specify how long after the last acquisition in your batch of `N_SHOTS` you would like to play the next Bayesian-update Rabi pulse.

## The Optimizer Logic (Emulated Coprocessor)

The definition of the experiment is only half of the deal. We also must implement the logic that runs on the coprocessor. This example uses an emulated coprocessor, where the compute kernel is written in Python.

For the sake of simplifying this demo, we do not implement a real estimator, and instead just source the newly "computed" amplitudes from a pre-defined list.

In [ ]:
class BayesianRabiKernel(EmulatedCoprocessorKernel):
    async def run(self, ctx):
        # "Fake" estimator for the next amplitude
        candidate_amplitude_generator = iter(np.linspace(0.1, 0.9, N_CYCLES))

        def calculate_next_amplitude(_current_measurement):
            return next(candidate_amplitude_generator)

        for _ in range(N_CYCLES):
            batch = [await ctx.streams["shots"].recv() for _ in range(N_SHOTS)]
            avg = float(np.mean([abs(shot.meas["rabi_shot"]) for shot in batch]))
            next_amplitude = calculate_next_amplitude(avg)

            # A real optimizer would also emit updated_waveform= here.
            # This is not supported in the emulated coprocessor yet.
            ctx.streams["updates"].emit(rabi_amplitude=float(next_amplitude))

## Compile and run the experiment

We map the experiment signals to logical signals in the DeviceSetup [as usual](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/05_experiment/concepts/index.html#experimental-signal-map). In addition, we also map the experiment optimizer to the emulated coprocessor. You can find all the available coprocessors in `setup.coprocessors`. Any physical coprocessors would also be listed there when available.

After we compile the experiment, we attached the `BayesianRabiKernel` as a payload the emulated coprocessor. We could have also done this directly in the experiment definiton, see LabOne Q DSL Constructs for Classical Processing.

In [ ]:
# Create the experiment
exp = bayesian_rabi()

# Map the experiment signals to logical signals in the DeviceSetup
exp.map_signal("drive", setup.logical_signal_groups["q0"].logical_signals["drive_line"])
exp.map_signal(
    "measure", setup.logical_signal_groups["q0"].logical_signals["measure_line"]
)
exp.map_signal(
    "acquire", setup.logical_signal_groups["q0"].logical_signals["acquire_line"]
)
# Map the experiment optimizer to the emulated coprocessor
exp.map_coprocessor("optimizer", emulated_coprocessor)

# Compile the experiment
compiled = session.compile(exp)
# Upload the payload to the experiment optimizer
compiled.set_coprocessor_payload("optimizer", b"__main__:BayesianRabiKernel")

# Run the experiment
results = session.run(compiled)

## Inspecting the Logged Variables in the Results

In the results, we can trace the decisions taken by the coprocessor:

In [ ]:
chosen = results.variable_results["chosen_amplitude"]
acquired = results.acquired_results["rabi_shot"]

print("log shape :", chosen.data.shape, "axes:", chosen.axis_name)
print("acq shape :", acquired.data.shape, "axes:", acquired.axis_name)
print("amplitudes:", chosen.data)

In [ ]:
fig, (top, bottom) = plt.subplots(2, 1, sharex=True, figsize=(7, 5))

top.step(np.arange(N_CYCLES), chosen.data, where="post")
top.set_ylabel("chosen amplitude")
top.set_title("amplitude picked by the optimizer")

bottom.plot(np.arange(N_CYCLES), np.abs(acquired.data).mean(axis=1), marker="o")
bottom.set_xlabel("optimizer cycle")
bottom.set_ylabel("mean |readout|")
bottom.set_title("measured value")

fig.tight_layout()
plt.show()

## References

1. <a id="ref1"></a>Stace, T. M. et al. Optimized Bayesian system identification in quantum devices. [Phys. Rev. Applied 21, 014012 (2024)](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.21.014012).
2. <a id="ref2"></a>Qian, P. et al. Fast quantum calibration using Bayesian optimization with state parameter estimator for non-Markovian environment. Preprint at [https://arxiv.org/abs/2205.12929](https://arxiv.org/abs/2205.12929) (2022).
3. <a id="ref3"></a>Berritta, F. et al. Real-time adaptive tracking of fluctuating relaxation rates in superconducting qubits. [Phys. Rev. X 16, 011025 (2026)](https://journals.aps.org/prx/abstract/10.1103/gk1b-stl3).
4. <a id="ref4"></a>Berritta, F. et al. Efficient qubit calibration by binary-search Hamiltonian tracking. [PRX Quantum 6, 030335 (2025)](https://journals.aps.org/prxquantum/abstract/10.1103/77qg-p68k).